# arXiv Nhóm 1 — Crawl & Benchmark

One entrypoint for the M1 CPU-only Colab crawl and benchmark. Final mode covers all four assigned ranges (20,000 IDs), every discovered version, and Semantic Scholar references. Sample mode is for validation only.

Drafted with assistance from OpenAI Codex; follows the group's AI disclosure convention.

**Before running:** select CPU/None hardware accelerator and coordinate one active group crawler. HF mode uses the configured private Dataset and HF_TOKEN from Colab Secrets; no Drive output mount is needed. Drive mode remains available as an explicit fallback. Do not run another arXiv crawler in parallel.

Drive-only initialization requires an explicit team-marker flag; HF uses the existing private Dataset. New runs never delete old data. After a runtime reset, select RESUME with the same run ID/settings. A four-hour session budget stops between chunks; Colab can terminate earlier. A missing timing summary is disclosed, never silently counted as zero.


HF mode: store HF_TOKEN in Colab Secrets and enable notebook access. The existing Dataset must be private and writable. Default configuration is a 100-ID smoke, not the final benchmark. Crawl files stay raw, sharded only on HF; resume restores the flat local corpus layout. Only one operator may run; coordinate handoff in group chat after sync and final report publication complete. The HF shift marker is advisory, not an atomic distributed lock. Do not claim a successful private-HF smoke until this notebook has run against the real Dataset.


In [ ]:
# Edit this cell. Use a NEW run ID for each new experiment.
MODE = "sample"  # "sample" for a trial; sample results are never final
RUN_ID = "m1-hf-smoke-dang-100-01"
STORAGE_BACKEND = "huggingface"  # or "drive"
HF_REPO_ID = "PancakeDang/arxiv-nhom1-introds"
RANGES = ("dang",)
OFFSET = 0
COUNT = 100  # final: MODE="final", all four ranges, COUNT=None
CHUNK_SIZE = 100
WITH_REFERENCES = True
SNAPSHOT_PATH = "/content/arxiv-metadata-oai-snapshot.json"
CHECKPOINT_S = 300
SESSION_BUDGET_S = 4 * 60 * 60
RESUME = False  # True in a new runtime after a saved run/timeout
SHIFT_OWNER = "dang"  # who is operating the shared crawl account now
TAKEOVER = False  # only after confirming the previous runtime has stopped
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"
TEAM_ID = "arxiv-nhom1"
INITIALIZE_TEAM_FOLDER = False  # True once, after checking the intended Drive folder
CODE_REF = "69d8382ad4704a7d72df8337de2009d9ce946c17"  # immutable snapshot-first implementation
REPO = "https://github.com/DankDaPancake/CodeRepo.git"

In [ ]:
# Prevent a new checkout from being mislabeled while Python keeps old src modules.
import sys

if any(name == "src" or name.startswith("src.") for name in sys.modules):
    raise RuntimeError(
        "Restart runtime before setup; cached pipeline modules may belong to another revision."
    )

## Authorize storage and resolve the saved code revision

HF mode reads HF_TOKEN from Colab Secrets and checks the existing private Dataset. Tokens are never written into the notebook, manifests or chunk payloads. On resume, the saved implementation revision is loaded from the same Dataset/run. Copy the unchanged Kaggle input into /content before benchmarking. Drive mode mounts the selected Google account only when explicitly selected.


In [ ]:
import json
import os
import subprocess
import sys
import uuid
from pathlib import Path

from google.colab import userdata

if STORAGE_BACKEND == "huggingface":
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN").strip()
    except Exception as exc:
        raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable notebook access") from exc
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.34,<2"], check=True
    )
    os.environ["HF_XET_CACHE"] = "/content/m1-hf-xet-cache"
    from huggingface_hub import HfApi, hf_hub_download

    hf_api = HfApi(token=os.environ["HF_TOKEN"])
    hf_info = hf_api.repo_info(HF_REPO_ID, repo_type="dataset")
    if not hf_info.private:
        raise ValueError("Expected the configured private Dataset")
    print("Private Dataset accessible:", HF_REPO_ID)
    print("Write permission will be checked by the shift claim before crawling")
    DRIVE = Path("/content/m1-hf-staging") / uuid.uuid4().hex
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path(DRIVE_DIR)
    if not DRIVE.is_dir():
        raise FileNotFoundError("Create/check the intended team Drive folder before running")

revision = CODE_REF
if RESUME:
    if STORAGE_BACKEND == "huggingface":
        manifest_path = hf_hub_download(
            HF_REPO_ID,
            f"runs/{RUN_ID}/MANIFEST.json",
            repo_type="dataset",
            token=os.environ["HF_TOKEN"],
        )
    else:
        manifest_path = DRIVE / "runs" / RUN_ID / "MANIFEST.json"
    saved = json.loads(Path(manifest_path).read_text())
    revision = saved["code_revision"]
repo_dir = Path("/content/CodeRepo")
if not (repo_dir / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO, str(repo_dir)], check=True)
subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", revision], check=True)
subprocess.run(["git", "-C", str(repo_dir), "checkout", "--detach", "FETCH_HEAD"], check=True)
os.chdir(repo_dir)
REVISION = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Implementation revision:", REVISION)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "src/requirements.txt"], check=True
)
if not Path("/usr/bin/time").is_file():
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "time"], check=True)
for name in ("ARXIV_CONTACT_EMAIL", "S2_API_KEY"):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value.strip()
    print(name, "configured" if value else "not configured")
LOCAL_ROOT = Path("/content/m1-local") / uuid.uuid4().hex
if SNAPSHOT_PATH and not Path(SNAPSHOT_PATH).is_file():
    raise FileNotFoundError("Copy the Kaggle snapshot to SNAPSHOT_PATH before benchmarking")

## Preflight and exact scope

Check the displayed ID count, code revision, CPU runtime and storage destination before the long-running cell. A 100-ID Dang sample spans 2604.20535–2604.20634. Final mode rejects partial ranges or references disabled.


In [ ]:
from dataclasses import asdict

import psutil

from src.benchmark.checkpoints import prepare_drive
from src.benchmark.runner import BenchmarkConfig, plan_chunks, run_benchmark

if STORAGE_BACKEND == "drive":
    prepare_drive(DRIVE, TEAM_ID, initialize=INITIALIZE_TEAM_FOLDER)
cfg = BenchmarkConfig(
    run_id=RUN_ID,
    mode=MODE,
    ranges=tuple(RANGES),
    offset=OFFSET,
    count=COUNT,
    chunk_size=CHUNK_SIZE,
    references=WITH_REFERENCES,
    snapshot=SNAPSHOT_PATH,
    storage_backend=STORAGE_BACKEND,
    hf_repo_id=HF_REPO_ID if STORAGE_BACKEND == "huggingface" else None,
    checkpoint_s=CHECKPOINT_S,
)
chunks = plan_chunks(cfg)
print(asdict(cfg))
print("Selected IDs:", sum(len(c.ids) for c in chunks), "chunks:", len(chunks))
print("CPU count:", psutil.cpu_count(), "RAM MB:", psutil.virtual_memory().total / 1e6)
print("Storage:", STORAGE_BACKEND, HF_REPO_ID if STORAGE_BACKEND == "huggingface" else DRIVE)
subprocess.run(["lscpu"], check=True)
subprocess.run(["df", "-h", "/content"], check=True)

## Crawl, checkpoint and benchmark

Each chunk uses the existing crawler under the process-tree profiler and `/usr/bin/time -v`. Periodic checkpoints run inside the measured crawler. Post-chunk sync is reported separately. Retry and backoff remain in wall time; do not add them again. Setup/install and offline gaps between sessions are excluded from pipeline time.

Checkpoint target: five minutes at completed-version boundaries. A large version or failed Drive upload can exceed that interval. Only completed storage checkpoints survive a destroyed runtime. A file being downloaded may need to restart after runtime loss; live HTTP Range recovery is not a cross-runtime promise.

**To end a shift:** normally let the session budget stop the run. If using Stop, wait until processes have stopped and the final checkpoint completes. After an abrupt disconnect, confirm the old runtime is dead before explicit takeover. Storage shift markers are advisory, not atomic distributed locks.


With SNAPSHOT_PATH set, paper discovery uses Kaggle first. Only missing/invalid records and historical version titles/authors require arXiv metadata requests. The snapshot fixes version coverage as of its release; do not claim it reflects later updates. A verified 20,000-ID subset is built once per run and restored with checkpoints. Setup hashing, index preparation, and crawl timing are reported separately; index preparation is included in session resource measurements. Download and unzip Kaggle to local /content before running; do not place the multi-GB input in the corpus or Drive checkpoint.


In [ ]:
result = run_benchmark(
    cfg,
    drive_root=DRIVE,
    local_root=LOCAL_ROOT,
    revision=REVISION,
    team_id=TEAM_ID,
    owner=SHIFT_OWNER,
    resume=RESUME,
    takeover=TAKEOVER,
    session_budget_s=SESSION_BUDGET_S,
)
print(json.dumps(result, indent=2))
print("Local corpus for M2:", LOCAL_ROOT / RUN_ID / "data/corpus")
print(
    "Durable raw corpus:",
    f"https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/runs/{RUN_ID}/data/corpus"
    if STORAGE_BACKEND == "huggingface"
    else DRIVE / "runs" / RUN_ID / "data/corpus",
)

## Inspect measurements and download

Full-session RSS includes the runner/notebook kernel baseline and all descendants. Session wall includes restore, crawling, references and final corpus sync. Samples target independent Drive publication every 30 seconds; corpus checkpoints still target 300 seconds. Peaks are sampled observations.

If a runtime dies before finalization, its last durable sample establishes a lower bound. Exact end-to-end and mean-time metrics are omitted until evidence is complete; do not replace missing time with zero. Corpus completeness and timing completeness are separate.

`paper_attempt_timings.csv` retains admission-to-readiness lifecycle latency for each paper/attempt, including batch queue waiting and required references. These latencies overlap. Use session wall / required IDs for amortized mean time. `source_work_s` measures version work separately.

Use a new run ID for this measurement revision. Setup, final evidence re-export/download and offline gaps are excluded. Raw corpus stays mirrored as real folders under `runs/<RUN_ID>/data/corpus` on Drive, ready for M2 to read directly; the downloaded ZIP only ever contains measurements, logs and QA, never the corpus itself.

In [ ]:
import pandas as pd
from google.colab import files
from IPython.display import display

from src.benchmark.runner import export_evidence, export_rescue

local = LOCAL_ROOT / RUN_ID
try:
    archive = export_evidence(cfg, local)
except Exception as exc:
    print("Statistics export failed; creating raw local rescue:", exc)
    archive = export_rescue(local)
for name in ("coverage.json", "timing_audit.json"):
    p = local / "work" / name
    if p.exists():
        print(name, p.read_text())
audit = local / "book_csv/measurement_audit.json"
if audit.exists():
    print("Measurement audit:", audit.read_text())
errors = local / "work/cleanup_errors.json"
if errors.exists():
    print("Cleanup diagnostics:", errors.read_text())
for name in ("benchmark_summary.csv", "corpus_summary.csv"):
    p = local / "book_csv" / name
    if p.exists():
        print(name)
        display(pd.read_csv(p))
p = local / "book_csv/paper_attempt_timings.csv"
if p.exists():
    print("Per-paper attempts (first 10; all rows in ZIP)")
    display(pd.read_csv(p).head(10))
print("Evidence:", archive)
files.download(str(archive))

## Resume in a fresh runtime

Keep RUN_ID and all dataset settings unchanged; set RESUME=True. Leave INITIALIZE_TEAM_FOLDER=False. The setup cell loads the saved code revision. Restore re-verifies every mirrored file's hash against the Drive `INDEX.json` before reuse, then skips only chunks passing corpus/journal/reference QA. A kill mid-upload only ever leaves a stray temp file beside the real one; `INDEX.json` is updated last and always points at a fully verified state, so restore ignores the stray file automatically.

If another shift marker remains after timeout, confirm the previous runtime has actually stopped, then set TAKEOVER=True for this handoff. Do not use another account/runtime to run a second crawler concurrently. The final report records sessions separately and does not claim one uninterrupted runtime.

HF resume uses the last completed checkpoint commit, never an in-progress main tree. Keep HF_REPO_ID, RUN_ID, snapshot bytes/path, and dataset settings unchanged. Uploads may span multiple commits; LATEST.json is published only after the index is read back at the data commit. Empty PDF-only version folders are recreated from INDEX.json. HF paths add month/sequence shards; local M2 paths remain data/corpus/<paper>/vN.
